

- [LCEL+structured outputの使い方](https://qiita.com/ssc-ymuramatsu/items/120e87d4a4751ab608f9)

In [8]:

import os
import csv
import pandas as pd
from langchain_google_genai import GoogleGenerativeAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.output_parsers import PydanticOutputParser
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
from typing import List
from dotenv import load_dotenv
import time
import json
import numpy as np
from logging import getLogger, StreamHandler, INFO,DEBUG
logger = getLogger(__name__)
handler = StreamHandler()
handler.setLevel(DEBUG)
logger.setLevel(DEBUG)
logger.addHandler(handler)

# 環境変数からAPIキーを読み込む
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
NUMBER_OF_EXAMPLES = os.getenv("NUMBER_OF_EXAMPLES", default=5)

# 構造化出力のためのPydanticモデル
class ExampleSentences(BaseModel):
    examples: List[str] = Field(...,description="List of 5 example sentences for the given word")

from langchain_core.rate_limiters import InMemoryRateLimiter

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.1,  # <-- Super slow! We can only make a request once every 10 seconds!!
    check_every_n_seconds=0.1,  # Wake up every 100 ms to check whether allowed to make a request,
    max_bucket_size=10,  # Controls the maximum burst size.
)

# Gemini APIの初期化
llm = GoogleGenerativeAI(model="gemini-2.0-flash", google_api_key=GOOGLE_API_KEY,rate_limiter=rate_limiter)

# output_parser = StrOutputParser(ExampleSentences)
output_parser = PydanticOutputParser(pydantic_object=ExampleSentences)

# format_instructions を生成
format_instructions = output_parser.get_format_instructions()


# 例文生成用のプロンプトテンプレート
example_prompt = PromptTemplate.from_template(
"""Generate 5 example sentences using the English word "{word}".
    
    Requirements:
    - Create natural sentences that a native speaker would use in everyday conversation or writing
    - Sentences should demonstrate typical usage of the word, including famous collocations or idioms
    - Vary the sentence structures and contexts
    - Make sentences clear and straightforward, yet not too short
    - Include both conversational and formal contexts
    - If the word has multiple meanings, provide sentences for each meaning
    
    {format_instructions}
    """
)
# format_instructionsをテンプレートに挿入
example_prompt = example_prompt.partial(format_instructions=format_instructions)

# Chain
chain = example_prompt | llm | output_parser

def load_input_csv(input_file:str):
    # 入力CSVの読み込み(重複なしの想定)
    try:
        df = pd.read_csv(input_file,dtype={'number': int, 'word': str}, na_filter=False)
    except Exception as e:
        print(f"CSVファイルの読み込みエラー: {e}")
        raise ValueError(f"CSVファイルの読み込みエラー: {e}")
    return df 
    

def check_existing_words(output_file:str):
    """既存の出力ファイルをチェックし、完全に処理された単語を取得する"""
    
    fully_processed_words = set()
    word_example_dict = {}
    
    if not os.path.exists(output_file):
        return fully_processed_words, word_example_dict
    
    try:
        existing_df = pd.read_csv(output_file,dtype={'number': int, 'word': str, 'example': str},na_filter=False)
        # NaNの処理を改善（pandasの標準的な方法でNaNをチェック）
        for word, group in existing_df.groupby('word'):
            examples = group['example'].tolist()
            # NaNでない例文が存在するかチェック
            valid_examples = [ex for ex in examples if pd.notna(ex) and ex != ""]
            print(f"単語: {word}, 有効な例文: {valid_examples}")
            
            if valid_examples:
                fully_processed_words.add(word)
                word_example_dict[word] = valid_examples
        print(f"既存の出力ファイルから{len(fully_processed_words)}件のデータを読み込みました。")
    except Exception as e:
        print(f"既存の出力ファイルの読み込みエラー: {e}")
    return fully_processed_words, word_example_dict
    

def generate_example(word:str)->List[str]:
    """単語に対して例文を生成する"""
    # 例文の生成
    output = chain.invoke({"word": word})
    # 出力をパース
    examples:List[str] = output.examples
    
    # 5つの例文がない場合は足りない分を空欄で補完
    while len(examples) < 5:
        examples.append("")
    
    # 最初の5つだけを使用
    examples = examples[:5]
    return examples

def convert_to_long_format(results:List[dict], word_id_dict:dict)->List[dict]:
    """結果をロング形式に変換する"""
    long_format_results = []
    for index, row in word_id_dict.iterrows():
        word = row['word']
        word_id = row['number']
        
        if word in results:
            for example in results[word]:
                long_format_results.append({
                    'number': word_id,
                    'word': word,
                    'example': example
                })
    return long_format_results


def process_csv(input_file, output_file):
    """入力CSVファイルを処理し、各単語に例文を追加して新しいCSVに保存する"""
    df = load_input_csv(input_file)
    
    # 既存の出力ファイルをチェック（続きから処理するため）
    fully_processed_words,word_example_dict = check_existing_words(output_file)
    
    # 各単語を処理
    for index, row in df.iterrows():
        word_id:int = row["number"]
        word = row['word']
        
        # 既に処理済みの単語はスキップ
        if word in fully_processed_words:
            print(f"スキップ: {word} (既に処理済み)")
            continue        
        print(f"処理中: {word} ({index+1}/{len(df)})")
        
        try:
            # 例文を生成
            examples = generate_example(word)
            # 結果を辞書に追加または更新
            word_example_dict[word] = examples
        except Exception as e:
            print(f"単語「{word}」の処理中にエラーが発生: {e}")
            # エラーが発生した場合も、空の例文で結果を追加して保存
            word_example_dict[word] = [""] * 5

        # 形式を変形し，ファイルに保存
        long_format_results = convert_to_long_format(word_example_dict, df)
        pd.DataFrame(long_format_results).to_csv(output_file, index=False, encoding="utf-8")
        print(f"  → {word}の処理完了。中間結果を保存しました。")
        # APIリクエスト制限を考慮して少し待機
        time.sleep(5)

    print(f"処理が完了しました。結果は {output_file} に保存されています。")



In [9]:
# 使用例
if __name__ == "__main__":
    input_file = "lv9.csv"  # 入力ファイル名
    output_file = "lv9_with_examples.csv"  # 出力ファイル名
    
    process_csv(input_file, output_file)

単語:  phonetic, 有効な例文: []
単語: Messrs., 有効な例文: []
単語: abbreviate, 有効な例文: ["Please abbreviate 'United States of America' to 'USA' when space is limited.", "In academic writing, you should not abbreviate words like 'because' or 'that is'.", "The doctor decided to abbreviate the patient's hospital stay after seeing significant improvement.", 'Can you abbreviate this long list of ingredients for the recipe instructions?', "It's common to abbreviate titles like 'Doctor' to 'Dr.' in formal correspondence."]
単語: aborable, 有効な例文: ['The puppy was so aborable with its floppy ears and clumsy paws.', 'Her aborable habit of humming while she works always brings a smile to my face.', 'Despite his gruff exterior, he has an aborable soft spot for his grandchildren.', "The children put on an aborable performance of 'Twinkle Twinkle Little Star' at the school concert.", 'I found an aborable vintage dress at the flea market, perfect for the summer.']
単語: abort, 有効な例文: ['The mission had to be aborted due to

In [55]:
fully_processed_words = set()
df = pd.read_csv(output_file)
results = df.to_dict('records')
for record in results:
    word = record['word']
    # 少なくとも一つの例文が空欄なら再処理対象
    if not record.get(f'example', ''):
        print(f"不完全な結果が見つかりました: {word} - 再処理します")
    else:
        fully_processed_words.add(word)

fully_processed_words

{'elasticity',
 'alternately',
 'slack',
 'partition',
 'constable',
 'tart',
 'adherence',
 'thermal',
 'barter',
 'dubious',
 'reenter',
 'objectivity',
 'tactics',
 'monotony',
 'boarder',
 'premature',
 'matrix',
 'akin',
 'showroom',
 'thump',
 'recur',
 'villain',
 'pinnacle',
 'supplementary',
 'sewer',
 'synthesize',
 'remnant',
 'sturdy',
 'intrigue',
 'respectfully',
 'grandeur',
 'archbishop',
 'taint',
 'mobilize',
 'insincere',
 'maximize',
 'bran',
 'tasteless',
 'spinning',
 'repute',
 'robin',
 'humanist',
 'distortion',
 'feudal',
 'interviewee',
 'astray',
 'monologue',
 'rascal',
 'neutrality',
 'poke',
 'redundancy',
 'breeder',
 'rectangle',
 'deter',
 'subscriber',
 'bomber',
 'reservoir',
 'utopia',
 'barbarism',
 'soaked',
 'coherence',
 'repressive',
 'workable',
 'vocational',
 'repression',
 'gunpower',
 'simulation',
 'denounce',
 'broadcaster',
 'childbirth',
 'ailment',
 'turf',
 'unification',
 'trillion',
 'automate',
 'devoid',
 'loaded',
 'radius',
 'm

In [64]:
check_existing_words(output_file)

results = [{'number': 8001.0, 'word': 'acoustic', 'example': 'The band played an acoustic set at the small coffee shop, creating a very intimate atmosphere.'}, {'number': 8001.0, 'word': 'acoustic', 'example': 'This room needs some acoustic treatment to reduce the echo and reverberation.'}, {'number': 8001.0, 'word': 'acoustic', 'example': 'Acoustic guitars are often preferred for fingerstyle playing due to their warm and resonant tone.'}, {'number': 8001.0, 'word': 'acoustic', 'example': 'The scientists used sophisticated acoustic sensors to track the movement of whales in the ocean.'}, {'number': 8001.0, 'word': 'acoustic', 'example': 'Despite the damage to the speaker, the acoustic quality was surprisingly good.'}, {'number': 8002.0, 'word': 'animate', 'example': 'The lively music seemed to animate the crowd, and everyone started dancing.'}, {'number': 8002.0, 'word': 'animate', 'example': 'The animator used sophisticated software to animate the characters in the film.'}, {'number':

{'elasticity',
 'alternately',
 'slack',
 'partition',
 'constable',
 'tart',
 'adherence',
 'thermal',
 'barter',
 'dubious',
 'reenter',
 'objectivity',
 'tactics',
 'monotony',
 'boarder',
 'premature',
 'matrix',
 'akin',
 'showroom',
 'thump',
 'recur',
 'villain',
 'pinnacle',
 'supplementary',
 'sewer',
 'synthesize',
 'remnant',
 'sturdy',
 'intrigue',
 'respectfully',
 'grandeur',
 'archbishop',
 'taint',
 'mobilize',
 'insincere',
 'maximize',
 'bran',
 'tasteless',
 'spinning',
 'repute',
 'robin',
 'humanist',
 'distortion',
 'feudal',
 'interviewee',
 'astray',
 'monologue',
 'rascal',
 'neutrality',
 'poke',
 'redundancy',
 'breeder',
 'rectangle',
 'deter',
 'subscriber',
 'bomber',
 'reservoir',
 'utopia',
 'barbarism',
 'soaked',
 'coherence',
 'repressive',
 'workable',
 'vocational',
 'repression',
 'gunpower',
 'simulation',
 'denounce',
 'broadcaster',
 'childbirth',
 'ailment',
 'turf',
 'unification',
 'trillion',
 'automate',
 'devoid',
 'loaded',
 'radius',
 'm

In [152]:
input_file = "words.csv"  # 入力ファイル名
output_file = "words_with_examples.csv"  # 出力ファイル名
process_csv(input_file, output_file)


既存の出力ファイルから6件のデータを読み込みました。
スキップ: symposium (既に処理済み)
スキップ: broaden (既に処理済み)
スキップ: ecology (既に処理済み)
スキップ: circulate (既に処理済み)
スキップ: corpse (既に処理済み)
スキップ: convey (既に処理済み)
処理中: nan (7/7)
  → nanの処理完了。中間結果を保存しました。
処理が完了しました。結果は words_with_examples.csv に保存されています。


In [ ]:
output = chain.invoke({"word": "symposium"})
output

ExampleSentences(examples=['The university is hosting a symposium on climate change next month, featuring leading scientists from around the world.', "I'm presenting my research at a symposium dedicated to advancements in artificial intelligence.", 'Attendance at the symposium is mandatory for all graduate students in the biology department.', "After the symposium, we'll be publishing a book containing all the presented papers and discussions.", 'She found the symposium intellectually stimulating, as it offered a diverse range of perspectives on the topic of urban planning.'])

In [ ]:
output.examples

['The university is hosting a symposium on climate change next month, featuring leading scientists from around the world.',
 "I'm presenting my research at a symposium dedicated to advancements in artificial intelligence.",
 'Attendance at the symposium is mandatory for all graduate students in the biology department.',
 "After the symposium, we'll be publishing a book containing all the presented papers and discussions.",
 'She found the symposium intellectually stimulating, as it offered a diverse range of perspectives on the topic of urban planning.']

In [42]:
existing_df = pd.read_csv(output_file)
results = existing_df.to_dict('records')
print(results)
processed_words = set(existing_df['word'].tolist())
processed_words

[{'word': 'symposium', 'example': 'The university is hosting a symposium on climate change next month, featuring leading scientists from around the world.'}, {'word': 'symposium', 'example': "I'm attending a symposium on artificial intelligence in medicine, hoping to learn about the latest advancements."}, {'word': 'symposium', 'example': 'The annual philosophy symposium always sparks lively debates and insightful discussions among the attendees.'}, {'word': 'symposium', 'example': 'She presented her research at the international symposium, receiving positive feedback from her peers.'}, {'word': 'symposium', 'example': 'Due to unforeseen circumstances, the symposium on Renaissance art has been postponed until further notice.'}, {'word': 'broaden', 'example': 'Traveling to different countries can significantly broaden your perspective on the world.'}, {'word': 'broaden', 'example': 'The company is offering training programs to broaden the skill set of its employees.'}, {'word': 'broaden

{'affirm',
 'allege',
 'amazement',
 'anonymous',
 'appendix',
 'apprehension',
 'arouse',
 'arrogant',
 'bankruptcy',
 'bowel',
 'broaden',
 'bruise',
 'cane',
 'chancellor',
 'circulate',
 'coincide',
 'commissioner',
 'comprise',
 'conspiracy',
 'continuation',
 'contradict',
 'contradiction',
 'corpse',
 'cripple',
 'cylinder',
 'deafen',
 'debut',
 'deceitful',
 'dictation',
 'distributer',
 'doom',
 'dose',
 'drastic',
 'ecology',
 'enhance',
 'enrich',
 'envelop',
 'excel',
 'expertise',
 'expressive',
 'fig',
 'fingerprint',
 'flake',
 'flap',
 'flush',
 'fury',
 'gardening',
 'goldfish',
 'heighten',
 'hush',
 'hypothesis',
 'illuminate',
 'incorporate',
 'indignation',
 'intake',
 'intensify',
 'intensive',
 'lavatory',
 'lecturer',
 'lengthen',
 'lessen',
 'lieutenant',
 'lighten',
 'lighthouse',
 'locomotive',
 'madly',
 'marsh',
 'merchandise',
 'metropolis',
 'metropolitan',
 'miner',
 'nationalize',
 'optimism',
 'plaster',
 'plumber',
 'precede',
 'preface',
 'prevailin